In [19]:
import sagemaker
import torch
from sagemaker.pytorch import PyTorch, PyTorchModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

In [2]:
USE_GPU = False

TRAIN_DEVICE = 'ml.g4dn.xlarge' if USE_GPU else 'ml.m5.large'
DEPLOY_DEVICE = 'ml.m5.large'

In [24]:
session = sagemaker.Session()
role = sagemaker.get_execution_role()

estimator = PyTorch( # the estimator object is just a wrapper
    role=role, # it can hold the identity
    hyperparameters = {
        'learning_rate': 0.05,
        'epochs': 20
    },
    session=session,
    entry_point='LinMod.py',
    source_dir='./src',
    py_version= 'py310',
    framework_version='2.0.0',
    instance_count=1,
    instance_type=TRAIN_DEVICE
)

In [25]:
estimator.fit(wait=True)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-04-10-18-40-43-190


2026-04-10 18:40:47 Starting - Starting the training job...
2026-04-10 18:41:02 Starting - Preparing the instances for training...
2026-04-10 18:41:26 Downloading - Downloading input data...
2026-04-10 18:42:06 Downloading - Downloading the training image.........
2026-04-10 18:43:48 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-10 18:43:51,072 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-10 18:43:51,072 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-10 18:43:51,073 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-10 18:43:51,082 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-10 18:43:51,119 sagemaker_pytorch_container.training INFO     Invoking user tra

In [26]:
model_data = estimator.model_data
print(model_data)

s3://sagemaker-us-east-1-407975137156/pytorch-training-2026-04-10-18-40-43-190/output/model.tar.gz


In [27]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type=DEPLOY_DEVICE,
    entry_point='inference.py',
    source_dir='src',
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-407975137156/pytorch-training-2026-04-10-18-40-43-190/output/model.tar.gz), script artifact (src), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-407975137156/pytorch-training-2026-04-10-18-44-30-972/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: pytorch-training-2026-04-10-18-44-30-972
INFO:sagemaker:Creating endpoint-config with name pytorch-training-2026-04-10-18-44-30-972
INFO:sagemaker:Creating endpoint with name pytorch-training-2026-04-10-18-44-30-972


------!

In [28]:
test_input = [10, 20, -5]

response = predictor.predict(test_input)
print(response)

[[3055.32421875], [5996.9228515625], [-1357.07373046875]]


In [23]:
%pip show sagemaker

Name: sagemaker
Version: 2.245.0
Summary: Open source library for training and deploying models on Amazon SageMaker.
Home-page: https://github.com/aws/sagemaker-python-sdk
Author: Amazon Web Services
Author-email: 
License: 
Location: /opt/conda/lib/python3.12/site-packages
Requires: attrs, boto3, cloudpickle, docker, fastapi, google-pasta, graphene, importlib-metadata, jsonschema, numpy, omegaconf, packaging, pandas, pathos, platformdirs, protobuf, psutil, pyyaml, requests, sagemaker-core, schema, smdebug-rulesconfig, tblib, tqdm, urllib3, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [29]:
role = sagemaker.get_execution_role()
print(role)

arn:aws:iam::407975137156:role/service-role/AmazonSageMaker-ExecutionRole-20260407T164091
